# LAB 09 — Baseline và Transformer NER

Notebook triển khai baseline trước Transformer, hiển thị token, `input_ids`, `attention_mask`, `word_ids`, labels, logits, predictions; so sánh entity-level Precision, Recall, F1 trên cùng test set.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Image
from ner_utils import ID_TO_LABEL, ROOT, build_records, load_conll2003, tokenize_and_align_labels
from baseline_ner import train_baseline, predict_baseline
from transformer_ner import train_transformer
from reporting import analyze_tokenization, save_all_results
from information_extraction import evaluate_information_extraction

## 1. Nạp dữ liệu và kiểm tra nhãn

In [ ]:
dataset = load_conll2003()
display(pd.DataFrame([{"split": name, "sentences": len(dataset[name])} for name in ("train", "validation", "test")]))
display(pd.DataFrame({"tokens": dataset["train"][0]["tokens"], "labels": [ID_TO_LABEL[int(x)] for x in dataset["train"][0]["ner_tags"]]}))

## 2. Baseline Logistic Regression

Đặc trưng gồm word, hình dạng, prefix, suffix và cửa sổ ngữ cảnh gần. Không dùng nhãn POS chuẩn làm input.

In [ ]:
baseline = train_baseline(dataset["train"], ROOT / "models" / "baseline.joblib")
baseline_predictions = predict_baseline(baseline, dataset["test"])
display(pd.DataFrame({"word": dataset["test"][0]["tokens"],
                      "gold": [ID_TO_LABEL[int(x)] for x in dataset["test"][0]["ner_tags"]],
                      "baseline": baseline_predictions[0]}).head(35))

## 3. Tokenization và alignment ở cấp độ subword

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-cased", use_fast=True)
sample = dataset["test"][0]
encoded = tokenize_and_align_labels({"tokens": [sample["tokens"]], "ner_tags": [sample["ner_tags"]]}, tokenizer)
word_ids = encoded.word_ids(batch_index=0)
display(pd.DataFrame({"token": tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]),
                      "input_ids": encoded["input_ids"][0], "attention_mask": encoded["attention_mask"][0],
                      "word_ids": word_ids, "labels": encoded["labels"][0]}).head(40))

## 4. Fine-tune pretrained Transformer

Dùng nhãn `-100` cho các vị trí bị ignore. Chọn best checkpoint theo validation F1. Giữ cấu hình cố định với seed 42.

In [ ]:
experiment = train_transformer(dataset=dataset, model_name="distilbert/distilbert-base-cased",
                               epochs=3, batch_size=8, max_length=256)
transformer_records = experiment["records"]
display(pd.DataFrame([{"sentence": r["id"], "words": len(r["words"])} for r in transformer_records]).head())

## 5. Quan sát logits và predictions

Logits có shape batch × sequence length × 9. Nhãn ở các subword bị ignore không được đưa vào đánh giá.

In [ ]:
logits = experiment["prediction_output"].predictions
if isinstance(logits, tuple):
    logits = logits[0]
first_input = experiment["tokenizer"](experiment["test"][0]["tokens"], is_split_into_words=True,
                                       truncation=True, max_length=256)
first_logits = logits[0]
first_predictions = first_logits.argmax(axis=-1)
num_visible = len(first_input["input_ids"])
display(pd.DataFrame({"token": experiment["tokenizer"].convert_ids_to_tokens(first_input["input_ids"]),
                      "input_ids": first_input["input_ids"],
                      "attention_mask": first_input["attention_mask"],
                      "word_ids": first_input.word_ids(),
                      "prediction": [ID_TO_LABEL[int(x)] for x in first_predictions[:num_visible]],
                      "max_logit": first_logits[:num_visible].max(axis=-1)}).head(40))
display(pd.DataFrame({"shape": [str(logits.shape)], "labels_shape": [str(experiment["prediction_output"].label_ids.shape)]}))

## 6. Experiment 2–5 — So sánh cùng test set

In [ ]:
baseline_test = predict_baseline(baseline, experiment["test"])
selected_baseline = [[predictions[index] for index in indices]
                     for predictions, indices in zip(baseline_test, experiment["retained_indices"])]
baseline_records = build_records(experiment["test"], selected_baseline, experiment["retained_indices"])
scores = save_all_results(baseline_records, transformer_records, dataset["train"], experiment["test"])
display(pd.read_csv(ROOT / "results.csv"))
display(pd.read_csv(ROOT / "entity_metrics.csv"))
display(pd.read_csv(ROOT / "artifacts" / "entity_length_metrics.csv"))

## 7. Biểu đồ và phân tích lỗi

In [ ]:
display(Image(filename=str(ROOT / "entity_length_analysis.png")))
display(Image(filename=str(ROOT / "confusion_matrix.png")))
display(pd.read_csv(ROOT / "artifacts" / "error_examples.csv").head(15))

## 8. Experiment 1 và Experiment 6 — Phân tích bổ sung

In [ ]:
analyze_tokenization(experiment["test"], experiment["tokenizer"], minimum=40)
display(pd.read_csv(ROOT / "artifacts" / "tokenization_analysis.csv").head(20))

## 9. IE — Trích xuất thông tin và đánh giá trên 30 câu

In [ ]:
ie_scores = evaluate_information_extraction(experiment["model"], experiment["tokenizer"],
                                              experiment["test"], limit=30)
display(pd.DataFrame([ie_scores]))
display(pd.read_csv(ROOT / "artifacts" / "ie_metrics.csv"))